In [13]:
import pandas as pd
import requests 
import json
import json
import dataclasses
from dataclasses import dataclass
from time import time

import pandas as pd
from kafka import KafkaProducer


In [14]:
# See API Sample data print
url = "https://api.tfl.gov.uk/Line/25/Arrivals"
response = requests.get(url)

data = response.json()

data

[{'$type': 'Tfl.Api.Presentation.Entities.Prediction, Tfl.Api.Presentation.Entities',
  'id': '55026671',
  'operationType': 1,
  'vehicleId': 'SK20BBX',
  'naptanId': '490000013D',
  'stationName': 'Bank Station  / Cornhill',
  'lineId': '25',
  'lineName': '25',
  'platformName': 'D',
  'direction': 'inbound',
  'bearing': '90',
  'tripId': '284749',
  'baseVersion': '20260313',
  'destinationNaptanId': '',
  'destinationName': 'Ilford, Hainault Street',
  'timestamp': '2026-03-26T11:31:06.2497063Z',
  'timeToStation': 947,
  'currentLocation': '',
  'towards': 'Aldgate Or Liverpool Street',
  'expectedArrival': '2026-03-26T11:46:53Z',
  'timeToLive': '2026-03-26T11:47:23Z',
  'modeName': 'bus',
  'timing': {'$type': 'Tfl.Api.Presentation.Entities.PredictionTiming, Tfl.Api.Presentation.Entities',
   'countdownServerAdjustment': '-00:00:02.5002713',
   'source': '2026-03-26T11:05:41.124Z',
   'insert': '2026-03-26T11:31:03.154Z',
   'read': '2026-03-26T11:31:00.642Z',
   'sent': '2026

In [15]:
# Get 10 lines from all lines for buses only

url = "https://api.tfl.gov.uk/Line/Mode/bus"
lines = requests.get(url).json()

line_ids = [line["id"] for line in lines]

print(line_ids[:10])

['1', '100', '101', '102', '103', '104', '105', '106', '107', '108']


In [16]:
all_data = []

for line_id in line_ids:
    url = f"https://api.tfl.gov.uk/Line/{line_id}/Arrivals"
    
    try:
        response = requests.get(url)
        data = response.json()
        all_data.extend(data)
    except:
        continue

In [17]:
# Keep only dicts
clean_data = [d for d in all_data if isinstance(d, dict)]

df_arrivals = pd.DataFrame(clean_data)

In [18]:
# Unnest timing column 
timing_df = pd.json_normalize(df_arrivals['timing'])

# rename columns to avoid confusion 
timing_df = timing_df.add_prefix('timing_')

#concat timing df with main df and drop previous timing column
timing_df = timing_df[['timing_source', 'timing_insert', 'timing_read', 'timing_sent', 'timing_received']].copy()

df_arrivals.drop(columns=['timing'], inplace=True)


line_data = pd.concat([df_arrivals, timing_df], axis=1)
line_data.columns

Index(['$type', 'id', 'operationType', 'vehicleId', 'naptanId', 'stationName',
       'lineId', 'lineName', 'platformName', 'direction', 'bearing', 'tripId',
       'baseVersion', 'destinationNaptanId', 'destinationName', 'timestamp',
       'timeToStation', 'currentLocation', 'towards', 'expectedArrival',
       'timeToLive', 'modeName', 'timing_source', 'timing_insert',
       'timing_read', 'timing_sent', 'timing_received'],
      dtype='str')

In [19]:
columns = ['id', 'operationType', 'vehicleId', 'naptanId', 'stationName',
       'lineId', 'lineName', 'platformName', 'direction', 'bearing', 'tripId',
       'baseVersion', 'destinationNaptanId', 'destinationName', 'timestamp',
       'timeToStation', 'currentLocation', 'towards', 'expectedArrival',
       'timeToLive', 'modeName', 'timing_source', 'timing_insert',
       'timing_read', 'timing_sent', 'timing_received'] 
line_data = line_data[columns]
line_data.head(2)

,id,operationType,vehicleId,naptanId,stationName,lineId,lineName,platformName,direction,bearing,...,currentLocation,towards,expectedArrival,timeToLive,modeName,timing_source,timing_insert,timing_read,timing_sent,timing_received
0,-1958493566,1,BP15OMB,490004733C,Canada Water Bus Station,1,1,C,inbound,145,...,,null,2026-03-26T11:34:12Z,2026-03-26T11:34:42Z,bus,2026-03-26T11:05:41.124Z,2026-03-26T11:31:13.145Z,2026-03-26T11:31:10.693Z,2026-03-26T11:30:47Z,0001-01-01T00:00:00Z
1,604812273,1,BV66VKA,490011538W,Reverdy Road,1,1,PH,inbound,277,...,,Elephant & Castle,2026-03-26T11:58:53Z,2026-03-26T11:59:23Z,bus,2026-03-26T11:05:41.124Z,2026-03-26T11:31:03.154Z,2026-03-26T11:31:00.642Z,2026-03-26T11:30:47Z,0001-01-01T00:00:00Z


In [20]:
# Clean up the data

line_data.dtypes

id                       str
operationType          int64
vehicleId                str
naptanId                 str
stationName              str
lineId                   str
lineName                 str
platformName             str
direction                str
bearing                  str
tripId                   str
baseVersion              str
destinationNaptanId      str
destinationName          str
timestamp                str
timeToStation          int64
currentLocation          str
towards                  str
expectedArrival          str
timeToLive               str
modeName                 str
timing_source            str
timing_insert            str
timing_read              str
timing_sent              str
timing_received          str
dtype: object

In [21]:
# Process to push to redpanda

@dataclass
class tlfArrival:
    id            : str
    operationType          : int
    vehicleId                : str
    naptanId                 : str
    stationName              : str
    lineId                   : str
    lineName                 : str
    platformName             : str
    direction                : str
    bearing                  : str
    tripId                   : str
    baseVersion              : str
    destinationNaptanId      : str
    destinationName          : str
    timestamp                : str
    timeToStation            : int
    currentLocation          : str
    towards                  : str
    expectedArrival          : str
    timeToLive               : str
    modeName                 : str
    timing_source            : str
    timing_insert            : str
    timing_read              : str
    timing_sent              : str
    timing_received          : str

def trip_from_row(row):
    return tlfArrival(
        id=str(row['id']),
        operationType=int(row['operationType']),
        vehicleId=str(row['vehicleId']),
        naptanId=str(row['naptanId']),
        stationName=str(row['stationName']),
        lineId=str(row['lineId']),
        lineName=str(row['lineName']),
        platformName=str(row['platformName']),
        direction=str(row['direction']),
        bearing=str(row['bearing']),
        tripId=str(row['tripId']),
        baseVersion=str(row['baseVersion']),
        destinationNaptanId=str(row['destinationNaptanId']),
        destinationName=str(row['destinationName']),
        timestamp=str(row['timestamp']),
        timeToStation=int(row['timeToStation']),
        currentLocation=str(row['currentLocation']),
        towards=str(row['towards']),
        expectedArrival=str(row['expectedArrival']),
        timeToLive=str(row['timeToLive']),
        modeName=str(row['modeName']),
        timing_source=str(row['timing_source']),
        timing_insert=str(row['timing_insert']),
        timing_read=str(row['timing_read']),
        timing_sent=str(row['timing_sent']),
        timing_received=str(row['timing_received'])

    )


def trip_serializer(trip):
    trip_dict = dataclasses.asdict(trip)
    return json.dumps(trip_dict).encode('utf-8')

def trip_deserializer(data):
    json_str = data.decode('utf-8')
    trip_dict = json.loads(json_str)
    return tlfArrival(**trip_dict)

In [22]:
server = "localhost:9092"

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=trip_serializer
)

topic_name = "tlf-arrivals"

In [23]:
t0 = time()

for _, row in line_data.iterrows():
    trip = trip_from_row(row)
    producer.send(topic_name, value=trip)

producer.flush()

t1 = time()

print(f'took {(t1 - t0):.2f} seconds')

took 4.47 seconds


In [24]:
from kafka import KafkaConsumer

# Correcting the variable names to match your producer's functions
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server], 
    auto_offset_reset='earliest', # Start from the beginning of the topic
    group_id='arrivals-group',
    value_deserializer=trip_deserializer # Changed from ride_deserializer to match your function
)

In [25]:
# You may need to install pyarrow or fastparquet: pip install pyarrow
line_data.to_parquet('tlf_arrivals.parquet', engine='pyarrow', compression='snappy')
print("Data saved to tlf_arrivals.parquet")

Data saved to tlf_arrivals.parquet


In [26]:
line_data.shape

12213